---
title: "RCA: Why My Oblivion Remastered Slow-Motion Combat Mod Died"
author: "Vahram Poghosyan"
date: "2026-10-17"
categories: ["Gaming", "Linux", "Modding"]
format:
  html:
    code-fold: true
jupyter: python3
include-after-body:
  text: |
    <script type="application/javascript" src="../../javascript/light-dark.js"></script>
---

# The Symptom

My Oblivion Remastered install (Steam, Proton, Arch Linux) has a few dozen mods. One of them — **Ultra Combat**, a combat overhaul that slows down time during fights (perfect-block slowmo, perfect-dodge slowmo, kill-cam slowmo, bow-jump bullet time) — quietly stopped working at some point. Every *other* mod appeared to work fine, which made it look like a classic mod conflict.

Spoiler: it wasn't a conflict at all. This post is the root-cause analysis of what actually happened, because the diagnosis turned out to be a neat little lesson in how modded Unreal Engine games are wired together under Proton.

# The Mod Stack

Oblivion Remastered mods come in three distinct flavors, and knowing which is which turned out to be the key to the whole investigation:

| Layer | How it loads | Examples from my setup |
|---|---|---|
| **`.esp` plugins** | Natively by the game engine, listed in `Content/Dev/ObvData/Data/Plugins.txt` | Balanced NPC Level Cap, Lumiere, Responsive Combat AI, house mods |
| **Pak mods** | Dropped into `Content/Paks/~mods/`, loaded by UE5 itself | Natural Bodies texture/mesh paks |
| **UE4SS Lua mods** | Injected via **UE4SS**, which itself loads through a proxy `dwmapi.dll` in `Binaries/Win64/` | **Ultra Combat**, NaturalBodyMorph, KnarzsDisrobeSpells, TesSyncMapInjector |

Ultra Combat lives at `Binaries/Win64/ue4ss/Mods/UltraCombat/scripts/main.lua`. All of its slow-motion features work the same way: a Lua hook fires on a combat event and writes `WorldSettings.TimeDilation`. No UE4SS, no time dilation.

# Ruling Out a Mod Conflict

First hypothesis: some other mod is fighting over `TimeDilation`. Easy to check — grep every other UE4SS Lua mod for time-dilation writes:

```bash
grep -rn -i "timedilation\|slomo\|slowmo" ue4ss/Mods/UltraPlusExtensions ue4ss/Mods/NaturalBodyMorph \
  ue4ss/Mods/KnarzsDisrobeSpells ue4ss/Mods/TesSyncMapInjector ue4ss/Mods/BPML_GenericFunctions
```

**Zero hits.** Nothing else touches time dilation. The prime suspect (UltraPlusExtensions, a graphics mod that aggressively sets engine CVars) only pokes at rendering settings like `r.SkylightIntensityMultiplier`.

So why did "all the other mods work"? Because the mods I could *see* working were `.esp` and pak mods — the two layers that don't go through UE4SS at all. That observation was the trap: it framed the problem as "one mod broke" when actually an entire loading layer had died.

# The Smoking Gun: a Stale Log

UE4SS rewrites `ue4ss/UE4SS.log` on **every** launch. Compare timestamps:

```text
UE4SS.log                 last written  2026-04-11 11:23   ← UE4SS last ran here
vkd3d-proton.cache        modified      2026-04-27 01:32   ← game played, no UE4SS
vkd3d-proton.cache.write  modified      2026-09-20 11:02   ← game played, no UE4SS
Steam lastlaunch                        2026-09-20          (localconfig.vdf)
```

The April 11 log is pristine — Ultra Combat loads, registers every hook, prints `All systems initialized successfully!`. But the game demonstrably ran on April 27 and September 20 (the vkd3d shader cache doesn't lie) *without UE4SS ever writing a log line*.

Conclusion: since late April, **UE4SS was never injected**. Ultra Combat didn't break — it was never loaded. And neither were NaturalBodyMorph, KnarzsDisrobeSpells' Lua half, or TesSyncMapInjector. I'd simply only noticed the missing combat slowmo.

# Root Cause: Clobbered Steam Launch Options

UE4SS injects itself via a **proxy `dwmapi.dll`** placed next to the game exe (verifiable with `strings` — it literally says `UE4SS Injection Proxy`). On Windows that's enough: the local DLL shadows the system one. Under Wine/Proton it is *not* enough — Wine prefers its own **built-in** `dwmapi` unless you explicitly override it:

```text
WINEDLLOVERRIDES="dwmapi=n,b" %command%
```

And here's what my Steam launch options actually contained when I looked:

```text
"/mnt/game-drive/SteamLibrary/steamapps/common/Oblivion Remastered/OblivionRemastered/Binaries/Win64"
```

A bare quoted directory path. No `%command%`, no `WINEDLLOVERRIDES`. Since there's no `%command%`, Steam just appends the string as an argument to the game exe, which ignores it. Net effect: nothing.

**Timeline reconstruction** (from file mtimes): everything worked on April 11. On the night of April 26–27 I re-downloaded UE4SS and updated Ultra Combat — and somewhere in that session the launch options got overwritten with a stray pasted path, destroying the `WINEDLLOVERRIDES` line. Every session since launched a vanilla (well, esp-modded) game. The September Proton update to 11.0-100 rebuilt the prefix registry but was a red herring — the override was never stored in the registry anyway.

# The Fix (Two Birds, One Line)

While fixing the override I also wanted **OBSE64** (`obse64_loader.exe`) back in the loop — a leftover `launch.sh` showed I'd used it before. Both fit in one launch-options line:

```bash
WINEDLLOVERRIDES="dwmapi=n,b" bash -c 'exec "${@/OblivionRemastered.exe/OblivionRemastered\/Binaries\/Win64\/obse64_loader.exe}"' -- %command%
```

How it works:

- `WINEDLLOVERRIDES="dwmapi=n,b"` — load the **n**ative (UE4SS proxy) `dwmapi.dll` first, fall back to **b**uilt-in. This resurrects UE4SS and every Lua mod.
- The `bash -c 'exec ...' -- %command%` wrapper takes the exact command Steam would run and rewrites its final argument, swapping the stock launcher (`Oblivion Remastered/OblivionRemastered.exe`) for `Binaries/Win64/obse64_loader.exe`. Proton then runs the OBSE loader, which starts the real game with OBSE injected.

Pre-flight checks that de-risked this:

- The bash substitution was tested standalone and produces the correct path.
- The game binary reports version **1.512.105.0**, exactly matching `obse64_1_512_105.dll` — OBSE is version-locked, so this match matters.
- No `Data/OBSE/Plugins` folder exists yet, so no current mod *depends* on OBSE — if a future game update breaks the version lock, only OBSE plugins would suffer.

**Verification after one launch:** fresh timestamp on `ue4ss/UE4SS.log`, and a new `obse64.log` next to the game exe.

# Lessons Learned

1. **"Everything else works" is weak evidence.** My mods spanned three loading mechanisms; the visible ones shared nothing with the broken one. Identify the *layer* a mod loads through before hunting conflicts.
2. **Logs that stop updating are louder than logs with errors.** The decisive clue wasn't an error message — it was a log file frozen in April while shader caches kept getting newer.
3. **Steam launch options are fragile state.** They live in `localconfig.vdf`, they're one careless paste away from destruction, and nothing warns you. If a Proton game needs `WINEDLLOVERRIDES`, keep the line written down somewhere.
4. **Under Proton, local DLL proxies need explicit overrides.** A `dwmapi.dll`/`dinput8.dll`-style injector sitting next to the exe does nothing until `WINEDLLOVERRIDES="<dll>=n,b"` is on the command.